# 第 4 课 · Cookie 与 Session

> **内核提醒（每课都要看）**：本 notebook 必须在 conda 环境 **ai_learn** 中运行。
> 右上角内核应显示 `Python 3.10.21 ('ai_learn')`；打开后从上到下逐格运行（Shift+Enter）。
> 若报 `ModuleNotFoundError`，先检查内核是否选成了 ai_learn。

> 本课只做一件事：搞懂服务器怎么“记住你”，以及 requests 里的 **Session** 怎么自动帮你带 Cookie。
> 只引入两个新概念：**Cookie**、**Session**。

In [1]:
# 每个 notebook 开头都要重新 import 这一课要用的库
import requests                                   # HTTP 请求库（每本 notebook 都要重新 import）
print('requests 版本：', requests.__version__)

requests 版本： 2.34.2


## 4.1 Cookie 是什么：游乐园门口盖的“手环”

**类比**：你去游乐园，第一次买票进门，保安在你手腕上盖个章 / 给个手环。之后你去玩任何项目，只要亮出手环，工作人员就知道“你买过全天票了”，不用再买一次。

**是什么**：HTTP 协议本身是**无状态**的——默认情况下，服务器收到一个请求，**完全不记得你之前来过**。Cookie 就是服务器第一次响应时塞给你的一小段数据（通过响应头 `Set-Cookie`），让你以后每次请求都带上它，它就凭这段数据认出你。

**为什么需要**：登录状态、购物车、“记住我”，全靠 Cookie 维持。没有 Cookie，你每打开一个页面都得重新登录一次。

## 4.2 手动带 Cookie 很麻烦，所以有了 Session

如果没有任何辅助：你要先发一次请求拿到 Cookie，再手动把它拼成字典放进下一次请求头里——繁琐、易错。

**requests 的 `Session`** 就是一个“会自己揣着手环”的请求器：

- 用 `s = requests.Session()` 创建一个；
- 之后用 `s.get(...)` 代替 `requests.get(...)`；
- **服务器发回来的 Cookie 会被 Session 自动存下来，你之后每次用 `s.get`，它自动带上，你完全不用管。**

下面用 httpbin 的两个演示接口验证（联网用 try/except 包好）：

- `/cookies/set?名字=值`：服务器“发手环”，告诉你 Cookie 已设置；
- `/cookies`：服务器“看手环”，回显它现在从你身上读到的 Cookie。

In [2]:
try:
    s = requests.Session()      # 创建一个会自动保管 Cookie 的“会话”

    # 第一次：让服务器给我们设置一个 Cookie，名叫 favorite，值是 python
    r1 = s.get('https://httpbin.org/cookies/set',
               params={'favorite': 'python'}, timeout=10)
    print('第一次设置 Cookie 后，服务器说：', r1.json())

    # 第二次：什么都不传，只问“你现在从我身上读到什么 Cookie？”
    r2 = s.get('https://httpbin.org/cookies', timeout=10)
    print('第二次请求时，Session 自动带上的 Cookie：', r2.json())
    # 你会看到 favorite=python 被自动带回去了——这就是手环的作用。
except Exception as e:
    print('这次没联网，不影响学习：', type(e).__name__, e)

第一次设置 Cookie 后，服务器说： {'cookies': {'favorite': 'python'}}


第二次请求时，Session 自动带上的 Cookie： {'cookies': {'favorite': 'python'}}


## 4.3 对比：不用 Session，服务器就“不认识你”

把上面的过程不用 Session 重做一遍：两次都是全新的 `requests.get`，互相不认识，看看第二次还能不能读到 Cookie：

In [3]:
try:
    # 第一次请求设置 Cookie，但我们“转头就走”，没存下来
    requests.get('https://httpbin.org/cookies/set',
                 params={'favorite': 'python'}, timeout=10)

    # 第二次是一个全新的请求，没有任何 Cookie 被带上
    r = requests.get('https://httpbin.org/cookies', timeout=10)
    print('不用 Session 时，服务器读到的 Cookie：', r.json())
    # 多半是空的 {}——因为这只手是“新的”，手腕上没有手环。
except Exception as e:
    print('这次没联网，不影响学习：', type(e).__name__, e)

不用 Session 时，服务器读到的 Cookie： {'cookies': {}}


> 日常爬虫经验：只要你要**登录后抓数据**，就一律用 `Session`。
> 它会把登录后的 Cookie 一直揣着，后面的请求自动都是“登录状态”。

## 4.4 小结（记住这三句）

1. **Cookie** 是服务器发给你的“手环/小纸条”，用来在无状态的 HTTP 里认出你。
2. **`requests.Session()`** 会自动保存并在后续请求中回传 Cookie，你不用手动管。
3. 要保持登录状态 / 连续会话，就用 `s.get(...)` 代替 `requests.get(...)`。

## 4.5 随堂小练习（自己动手，空格里写）

1. 创建一个 Session，先用 `s.get('https://httpbin.org/cookies/set', params={...})` 设置两个 Cookie（自己起名和值），再用 `s.get('https://httpbin.org/cookies')` 打印回显。
2. 打印 `s.cookies`，看看 Session 内部把 Cookie 存在了什么结构里。

In [4]:
# TODO：自己写代码完成上面两题（这是随堂小练，没有单独答案文件）


## 来源与延伸

- requests 官方文档（会话对象 Session）：https://requests.readthedocs.io/en/latest/user/advanced/#session-objects
- MDN Cookie 介绍：https://developer.mozilla.org/zh-CN/docs/Web/HTTP/Cookies
- httpbin Cookie 接口：https://httpbin.org/#/Cookies
- 下一课：超时、重试，以及中文网页乱码怎么解决。